# ⚖️ GeoJusticia CDMX — Bases de Datos e Infraestructura del Proyecto

> *"El derecho a la salud no termina en la puerta, comienza en la banqueta."*

## ¿Qué es GeoJusticia CDMX?

GeoJusticia es una plataforma de **auditoría urbana ciudadana** que cuantifica el **Índice de Injusticia Espacial (IIN)**: el riesgo que el Estado impone a sus ciudadanos en el trayecto hacia sus servicios públicos asignados — infraestructura, movilidad, seguridad, servicios básicos.

No medimos solo la calidad del servicio. Medimos la seguridad y dignidad del camino para acceder a él.

---

## Fuentes de datos del proyecto

| # | Dataset | Fuente | URL |
|---|---------|--------|-----|
| 1 | Carpetas de investigación FGJ | ADIP / FGJ CDMX | https://datos.cdmx.gob.mx/dataset/carpetas-de-investigacion-fgj-de-la-ciudad-de-mexico |
| 2 | WiFi gratuito en Postes C5 | C5 CDMX | https://datos.cdmx.gob.mx/dataset/wifi-gratuito-en-postes-del-c5 |
| 3 | GTFS estático transporte público | SEMOVI / ADIP | https://datos.cdmx.gob.mx/dataset/gtfs |
| 4 | Colonias IECM 2022 (polígonos) | IECM | https://datos.cdmx.gob.mx/dataset/colonias-del-iecm-2022 |
| 5 | Infraestructura física RI-6 (alumbrado) | INEGI / SIEG | https://sieg.cdmx.gob.mx/layers/geonode:infra_alumbrado |
| 6 | Quejas ciudadanas SUAC/Locatel | CDMX | Portal SUAC / 0311 / App CDMX |

---

## Contenido

1. [Dependencias y configuración](#s1)
2. [FGJ — Delitos de alto impacto](#s2)
3. [C5 WiFi — Infraestructura digital](#s3)
4. [GTFS — Red de transporte público](#s4)
5. [Colonias IECM — Geografía base](#s5)
6. [RI-6 — Alumbrado e infraestructura física](#s6)
7. [SUAC/Locatel — Quejas ciudadanas](#s7)
8. [Índice de Injusticia Espacial (IIN)](#s8)
9. [Sistema de peticiones y alertas automáticas](#s9)


---
## 1. Dependencias y configuración <a id='s1'></a>


In [ ]:
# !pip install geopandas duckdb pyarrow openpyxl shapely schedule

import pandas as pd
import geopandas as gpd
import numpy as np
import duckdb, json, uuid, sqlite3, smtplib, schedule, time
from datetime import datetime, timedelta
from pathlib import Path
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import warnings; warnings.filterwarnings('ignore')

print(f'pandas    {pd.__version__}')
print(f'geopandas {gpd.__version__}')
print(f'duckdb    {duckdb.__version__}')


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
ROOT   = Path('.')
BRONZE = ROOT / 'data/bronze'
SILVER = ROOT / 'data/silver'
GOLD   = ROOT / 'data/gold'
for p in [SILVER, GOLD]: p.mkdir(parents=True, exist_ok=True)

WGS84 = 'EPSG:4326'
UTM14 = 'EPSG:32614'
BBOX  = {'lat_min':19.05,'lat_max':19.65,'lon_min':-99.40,'lon_max':-98.93}
DIAS_ESCALACION = 15

ALCALDIAS_MAP = {
    'ALVARO OBREGON':'Alvaro Obregon','Alvaro Obregon':'Alvaro Obregon',
    'AZCAPOTZALCO':'Azcapotzalco',
    'BENITO JUAREZ':'Benito Juarez','Benito Juarez':'Benito Juarez',
    'COYOACAN':'Coyoacan','Coyoacan':'Coyoacan',
    'CUAJIMALPA DE MORELOS':'Cuajimalpa','Cuajimalpa de Morelos':'Cuajimalpa',
    'CUAUHTEMOC':'Cuauhtemoc','Cuauhtemoc':'Cuauhtemoc',
    'GUSTAVO A. MADERO':'Gustavo A. Madero',
    'IZTACALCO':'Iztacalco','IZTAPALAPA':'Iztapalapa',
    'LA MAGDALENA CONTRERAS':'Magdalena Contreras','Magdalena Contreras':'Magdalena Contreras',
    'MIGUEL HIDALGO':'Miguel Hidalgo','MILPA ALTA':'Milpa Alta',
    'TLAHUAC':'Tlahuac','Tlahuac':'Tlahuac','TLALPAN':'Tlalpan',
    'VENUSTIANO CARRANZA':'Venustiano Carranza','XOCHIMILCO':'Xochimilco',
}

# Directorio de responsables — reemplazar con correos reales en produccion
RESPONSABLES = {
    'Alvaro Obregon':      {'delegado':'delegado.ao@cdmx.gob.mx',      'director':'director.ao@cdmx.gob.mx'},
    'Azcapotzalco':        {'delegado':'delegado.azc@cdmx.gob.mx',     'director':'director.azc@cdmx.gob.mx'},
    'Benito Juarez':       {'delegado':'delegado.bj@cdmx.gob.mx',      'director':'director.bj@cdmx.gob.mx'},
    'Coyoacan':            {'delegado':'delegado.coy@cdmx.gob.mx',     'director':'director.coy@cdmx.gob.mx'},
    'Cuajimalpa':          {'delegado':'delegado.cua@cdmx.gob.mx',     'director':'director.cua@cdmx.gob.mx'},
    'Cuauhtemoc':          {'delegado':'delegado.cuh@cdmx.gob.mx',     'director':'director.cuh@cdmx.gob.mx'},
    'Gustavo A. Madero':   {'delegado':'delegado.gam@cdmx.gob.mx',     'director':'director.gam@cdmx.gob.mx'},
    'Iztacalco':           {'delegado':'delegado.izc@cdmx.gob.mx',     'director':'director.izc@cdmx.gob.mx'},
    'Iztapalapa':          {'delegado':'delegado.izp@cdmx.gob.mx',     'director':'director.izp@cdmx.gob.mx'},
    'Magdalena Contreras': {'delegado':'delegado.mc@cdmx.gob.mx',      'director':'director.mc@cdmx.gob.mx'},
    'Miguel Hidalgo':      {'delegado':'delegado.mh@cdmx.gob.mx',      'director':'director.mh@cdmx.gob.mx'},
    'Milpa Alta':          {'delegado':'delegado.ma@cdmx.gob.mx',      'director':'director.ma@cdmx.gob.mx'},
    'Tlahuac':             {'delegado':'delegado.tlh@cdmx.gob.mx',     'director':'director.tlh@cdmx.gob.mx'},
    'Tlalpan':             {'delegado':'delegado.tlp@cdmx.gob.mx',     'director':'director.tlp@cdmx.gob.mx'},
    'Venustiano Carranza': {'delegado':'delegado.vc@cdmx.gob.mx',      'director':'director.vc@cdmx.gob.mx'},
    'Xochimilco':          {'delegado':'delegado.xoc@cdmx.gob.mx',     'director':'director.xoc@cdmx.gob.mx'},
}

EMAIL_CONFIG = {
    'smtp_host':'smtp.gmail.com','smtp_port':587,
    'sender':'geojusticia.cdmx@gmail.com',
    'password':'TU_APP_PASSWORD_AQUI',
}

print('Configuracion lista.')


---
## 2. FGJ — Delitos de alto impacto <a id='s2'></a>

**Rol en GeoJusticia:** Mide el riesgo real del trayecto. Filtramos delitos que afectan directamente al ciudadano en vía pública: robo a transeúnte, lesiones intencionales, homicidio doloso.

**Fuente:** https://datos.cdmx.gob.mx/dataset/carpetas-de-investigacion-fgj-de-la-ciudad-de-mexico  
**Actualización:** Mensual | **Registros 2024:** ~138,000 carpetas

> ⚠️ Cifra negra estimada ~85% (INEGI ENSU). Medimos presencia institucional de delitos, no incidencia real.


In [ ]:
df_fgj = pd.read_csv(BRONZE / 'carpetasFGJ_2024.csv', low_memory=False)
print(f'Shape: {df_fgj.shape}')
print(f'Rango: {df_fgj.fecha_inicio.min()} → {df_fgj.fecha_inicio.max()}')
print(f'Nulos coords: {df_fgj.latitud.isna().sum():,} ({df_fgj.latitud.isna().mean()*100:.1f}%)')
print()
print('Categorias de delito:')
print(df_fgj['categoria_delito'].value_counts().to_string())


In [ ]:
df_fgj['fecha_inicio'] = pd.to_datetime(df_fgj['fecha_inicio'], errors='coerce')
df_fgj['fecha_hecho']  = pd.to_datetime(df_fgj['fecha_hecho'],  errors='coerce')
df_fgj['mes_num'] = df_fgj['mes_inicio'].map({
    'Enero':1,'Febrero':2,'Marzo':3,'Abril':4,'Mayo':5,'Junio':6,
    'Julio':7,'Agosto':8,'Septiembre':9,'Octubre':10,'Noviembre':11,'Diciembre':12
})
df_fgj['alcaldia_norm'] = df_fgj['alcaldia_hecho'].str.strip().str.upper().map(ALCALDIAS_MAP)

def clasificar_delito(d):
    d = str(d).upper()
    if any(p in d for p in ['TRANSEUNTE','LESIONES INTENCIONAL','HOMICIDIO DOLOSO','VIOLACION','DISPARO DE ARMA']):
        return 'peaton'
    if any(p in d for p in ['PASAJERO','METRO','MICROBUS','TAXI']):
        return 'transporte'
    return 'otro'

df_fgj['relevancia'] = df_fgj['delito'].apply(clasificar_delito)

df_geo = df_fgj.dropna(subset=['latitud','longitud']).copy()
df_geo = df_geo[
    df_geo.latitud.between(BBOX['lat_min'],BBOX['lat_max']) &
    df_geo.longitud.between(BBOX['lon_min'],BBOX['lon_max'])
]

COLS_FGJ = ['fecha_inicio','fecha_hecho','mes_num','delito','categoria_delito',
            'relevancia','alcaldia_norm','colonia_hecho','latitud','longitud']
gdf_fgj = gpd.GeoDataFrame(
    df_geo[COLS_FGJ],
    geometry=gpd.points_from_xy(df_geo.longitud, df_geo.latitud), crs=WGS84
).to_crs(UTM14)

gdf_fgj.to_parquet(SILVER / 'fgj_2024.parquet')
print(f'Silver FGJ: {len(gdf_fgj):,} registros')
print()
print('Por relevancia:')
print(gdf_fgj['relevancia'].value_counts().to_string())


---
## 3. C5 WiFi — Infraestructura digital <a id='s3'></a>

**Rol en GeoJusticia:** Proxy de inversión institucional en el espacio público. Su densidad disuade el crimen y mejora la percepción de seguridad en el trayecto. A más postes → menor IIN.

**Fuente:** https://datos.cdmx.gob.mx/dataset/wifi-gratuito-en-postes-del-c5  
**Registros:** 13,714 postes · 0 nulos · 16 alcaldías


In [ ]:
df_wifi = pd.read_excel(BRONZE / '07-2025-wifi_gratuito_en_postes-del-c5.xlsx')
print(f'Shape: {df_wifi.shape}  Nulos: {df_wifi.isnull().sum().to_dict()}')
print('Por alcaldia:')
print(df_wifi['alcaldia'].value_counts().to_string())


In [ ]:
df_wifi['alcaldia_norm'] = df_wifi['alcaldia'].map(ALCALDIAS_MAP).fillna(df_wifi['alcaldia'])
df_wifi = df_wifi.rename(columns={'id':'poste_id'})

gdf_wifi = gpd.GeoDataFrame(
    df_wifi[['poste_id','alcaldia_norm','latitud','longitud']],
    geometry=gpd.points_from_xy(df_wifi.longitud, df_wifi.latitud), crs=WGS84
).to_crs(UTM14)

gdf_wifi.to_parquet(SILVER / 'wifi_c5.parquet')
print(f'Silver WiFi: {len(gdf_wifi):,} postes — UTM 14N')


---
## 4. GTFS — Red de transporte público <a id='s4'></a>

**Rol en GeoJusticia:** Las paradas GTFS definen los nodos de llegada al espacio público. La distancia parada→destino define la "última milla" a pie. Más paradas → más visibilidad social → menor riesgo.

**Fuente:** https://datos.cdmx.gob.mx/dataset/gtfs  
**Archivo:** `stops.txt` | **Registros:** 11,362 paradas (Metro, Metrobús, RTP, Trolebús, Cablebús)


In [ ]:
df_stops = pd.read_csv(BRONZE / 'stops.txt')
print(f'Shape: {df_stops.shape}  Nulos: {df_stops.isnull().sum().to_dict()}')

def inferir_sistema(z):
    z = str(z).upper()
    if z.startswith('0300'): return 'Metrobus'
    if z.startswith('05'):   return 'Metro'
    if z.startswith('04'):   return 'RTP'
    if z.startswith('06'):   return 'Tren Ligero'
    if z.startswith('07'):   return 'Cablebus'
    if z.startswith('08'):   return 'Trolebus'
    return 'Otro'

df_stops['sistema']  = df_stops['zone_id'].apply(inferir_sistema)
df_stops['accesible'] = df_stops['wheelchair_boarding'].map({0:'sin_info',1:'accesible',2:'inaccesible'})
df_stops = df_stops[
    df_stops.stop_lat.between(BBOX['lat_min'],BBOX['lat_max']) &
    df_stops.stop_lon.between(BBOX['lon_min'],BBOX['lon_max'])
].copy()

gdf_stops = gpd.GeoDataFrame(
    df_stops[['stop_id','stop_name','sistema','accesible','stop_lat','stop_lon']],
    geometry=gpd.points_from_xy(df_stops.stop_lon, df_stops.stop_lat), crs=WGS84
).to_crs(UTM14)

gdf_stops.to_parquet(SILVER / 'stops_gtfs.parquet')
print(f'Silver GTFS: {len(gdf_stops):,} paradas')
print(gdf_stops['sistema'].value_counts().to_string())


---
## 5. Colonias IECM 2022 — Geografía base <a id='s5'></a>

**Rol en GeoJusticia:** Unidad geográfica mínima de análisis. Permite comparar vulnerabilidad entre colonias de la misma alcaldía con condiciones muy distintas.

**Fuente:** https://datos.cdmx.gob.mx/dataset/colonias-del-iecm-2022  
**Archivo:** `colonias_iecm2022_.shp` (~2,800 polígonos)


In [ ]:
# Ajustar ruta si es necesario
gdf_colonias = gpd.read_file(BRONZE / 'colonias_iecm_2022/colonias_iecm2022_.shp')
gdf_colonias = gdf_colonias.to_crs(UTM14)
print(f'Shape: {gdf_colonias.shape}  CRS: {gdf_colonias.crs}')
print('Columnas:', gdf_colonias.columns.tolist())

if 'alcaldia' in gdf_colonias.columns:
    gdf_colonias['alcaldia_norm'] = gdf_colonias['alcaldia'].str.strip().str.upper().map(ALCALDIAS_MAP)

gdf_colonias.to_parquet(SILVER / 'colonias_iecm.parquet')
print(f'Silver Colonias: {len(gdf_colonias):,} poligonos — UTM 14N')


---
## 6. RI-6 — Alumbrado e infraestructura física <a id='s6'></a>

**Rol en GeoJusticia:** Score de alumbrado = riesgo nocturno del trayecto. Una colonia sin luminarias tiene penalización máxima en el IIN.

Variable clave: `r_AlumP_Mn` (1=mejor → 5=peor, 0=sin datos)  
→ Se transforma en `vulnerabilidad_luz` normalizada [0.0, 1.0]

**Fuente:** INEGI / SIEG CDMX — https://sieg.cdmx.gob.mx/layers/geonode:infra_alumbrado  
**Archivo:** `ri_6.shp` | **CRS original:** EPSG:32614 (UTM 14N, no necesita reproyección)


In [ ]:
gdf_infra = gpd.read_file(BRONZE / 'infra_fisica/ri_6.shp')
print(f'Shape: {gdf_infra.shape}  CRS: {gdf_infra.crs}  Nulos: {gdf_infra.isnull().sum().sum()}')
print(f'AlumP_sum: min={gdf_infra.AlumP_sum.min():.0f} max={gdf_infra.AlumP_sum.max():.0f}')
print(f'r_AlumP_Mn: {gdf_infra.r_AlumP_Mn.value_counts().sort_index().to_dict()}')
print(f'Colonias sin alumbrado: {(gdf_infra.AlumP_sum==0).sum()}')


In [ ]:
COLS_RI6 = ['cve_col','colonia','alcaldia','AlumP_sum','ALUMP_mean',
            'r_AlumP_Mn','P_PorcElec','P_AguaPot','S_RI_6','C_RI_6','geometry']
gdf_ri6 = gdf_infra[COLS_RI6].copy()
gdf_ri6['alcaldia_norm'] = gdf_ri6['alcaldia'].str.strip().str.upper().map(ALCALDIAS_MAP)

# vulnerabilidad_luz: 0=bien iluminado, 1=sin luz
gdf_ri6['vulnerabilidad_luz'] = np.where(
    gdf_ri6['r_AlumP_Mn'] > 0,
    (gdf_ri6['r_AlumP_Mn'] - 1) / 4,
    1.0  # sin datos = maxima vulnerabilidad
)
# rezago_infra: normalizacion Min-Max
mn, mx = gdf_ri6['S_RI_6'].min(), gdf_ri6['S_RI_6'].max()
gdf_ri6['rezago_infra'] = (gdf_ri6['S_RI_6'] - mn) / (mx - mn)

gdf_ri6.to_parquet(SILVER / 'infra_ri6.parquet')
print(f'Silver RI-6: {len(gdf_ri6):,} colonias')
print(f'  vulnerabilidad_luz: [{gdf_ri6.vulnerabilidad_luz.min():.2f}, {gdf_ri6.vulnerabilidad_luz.max():.2f}]')


---
## 7. SUAC/Locatel — Quejas ciudadanas <a id='s7'></a>

**Rol en GeoJusticia:** La voz del ciudadano cuantificada. Las quejas de alumbrado, bacheo y vigilancia validan los scores calculados con datos oficiales. Alta queja + alto RI-6 = prioridad máxima.

**Fuente:** SUAC CDMX (Portal / App CDMX / WhatsApp 0311)  
**Periodo:** Enero–Febrero 2024 | **Registros:** 38,708 quejas · 16 alcaldías


In [ ]:
df_loc = pd.read_csv(BRONZE / 'filtrado_locatel.csv', low_memory=False)
df_loc['fecha_solicitud'] = pd.to_datetime(df_loc['fecha_solicitud'], errors='coerce')
print(f'Shape: {df_loc.shape}')
print('Canales:', df_loc['tipo_de_entrada'].value_counts().to_dict())
print('Estatus:', df_loc['estatus'].value_counts().to_dict())
print()
print('Top 15 temas:')
print(df_loc['tema_solicitud'].value_counts().head(15).to_string())


In [ ]:
TEMAS_INFRA = [
    'ALUMBRADO','BACHEO','PAVIMENTACION','SOLICITUD DE VIGILANCIA',
    'MANTENIMIENTO VIA PUBLICA','FALTA DE AGUA','FUGA DE AGUA',
    'MANTENIMIENTO SEMAFOROS','BALIZAMIENTO','DESAZOLVE',
    'RECOLECCION BASURA','MANTENIMIENTO PARQUE / AREA VERDE','LIMPIEZA VIA PUBLICA'
]

df_loc['alcaldia_norm'] = df_loc['alcaldia_catalogo'].map(ALCALDIAS_MAP).fillna(df_loc['alcaldia_catalogo'])
df_loc['estatus_simple'] = df_loc['estatus'].map({
    'CERRADO':'cerrado','ATENDIDO':'atendido',
    'AUTORIDAD RESPONSABLE':'pendiente','RECIBIDO':'pendiente'
})
df_loc['es_infra'] = df_loc['tema_solicitud'].isin(TEMAS_INFRA)
df_loc['colonia_catalogo'] = df_loc['colonia_catalogo'].fillna('Sin colonia')

df_loc_geo = df_loc.dropna(subset=['latitud','longitud']).copy()
df_loc_geo = df_loc_geo[
    df_loc_geo.latitud.between(BBOX['lat_min'],BBOX['lat_max']) &
    df_loc_geo.longitud.between(BBOX['lon_min'],BBOX['lon_max'])
]

df_loc_geo.to_parquet(SILVER / 'locatel_2024.parquet', index=False)
print(f'Silver Locatel: {len(df_loc_geo):,} quejas con coords')
print(f'  De infraestructura: {df_loc_geo.es_infra.sum():,}')
print()
print('Tasa de resolucion por tema:')
for tema in TEMAS_INFRA[:8]:
    sub = df_loc_geo[df_loc_geo.tema_solicitud==tema]
    if len(sub) > 50:
        tasa = (sub.estatus_simple=='cerrado').sum()/len(sub)*100
        print(f'  {tema:<40} {tasa:5.1f}%  (n={len(sub):,})')


---
## 8. Índice de Injusticia Espacial (IIN) <a id='s8'></a>

$$I_{IN} = \frac{\text{Delitos}_{500m}}{\ln(\text{WiFi C5}_{500m} + \text{Paradas GTFS}_{300m} + 1)} \times \text{VulnerabilidadLuz}$$

Normalizado [0, 100]. Aplica a **cualquier punto** de la ciudad, no solo hospitales:
escuelas, centros comunitarios, oficinas de gobierno, paradas de transporte.


In [ ]:
# Cargar capas Silver
gdf_fgj_s   = gpd.read_parquet(SILVER / 'fgj_2024.parquet')
gdf_wifi_s  = gpd.read_parquet(SILVER / 'wifi_c5.parquet')
gdf_stops_s = gpd.read_parquet(SILVER / 'stops_gtfs.parquet')
gdf_ri6_s   = gpd.read_parquet(SILVER / 'infra_ri6.parquet')
gdf_peaton  = gdf_fgj_s[gdf_fgj_s.relevancia=='peaton'].copy()

print('Capas Silver cargadas:')
print(f'  FGJ peaton:   {len(gdf_peaton):,}')
print(f'  WiFi C5:      {len(gdf_wifi_s):,}')
print(f'  GTFS stops:   {len(gdf_stops_s):,}')
print(f'  RI-6 colonias:{len(gdf_ri6_s):,}')


In [ ]:
# Puntos de interes para calcular IIN
# Reemplazar con los puntos reales de tu analisis
from shapely.geometry import Point

PUNTOS = [
    {'nombre':'Plaza de la Constitucion',    'alcaldia':'Cuauhtemoc',         'lat':19.4326,'lon':-99.1332},
    {'nombre':'Centro Comunitario Tepito',   'alcaldia':'Cuauhtemoc',         'lat':19.4412,'lon':-99.1205},
    {'nombre':'Mercado Jamaica',             'alcaldia':'Venustiano Carranza','lat':19.4063,'lon':-99.1163},
    {'nombre':'UAM Iztapalapa',              'alcaldia':'Iztapalapa',         'lat':19.3574,'lon':-99.0731},
    {'nombre':'Mercado Tlalpan',             'alcaldia':'Tlalpan',            'lat':19.2980,'lon':-99.1527},
    {'nombre':'Plaza Garibaldi',             'alcaldia':'Cuauhtemoc',         'lat':19.4409,'lon':-99.1383},
    {'nombre':'Centro Comunitario Pedregal', 'alcaldia':'Tlalpan',            'lat':19.2510,'lon':-99.1870},
    {'nombre':'Parque Alameda Central',      'alcaldia':'Cuauhtemoc',         'lat':19.4358,'lon':-99.1457},
]

df_p = pd.DataFrame(PUNTOS)
gdf_p = gpd.GeoDataFrame(df_p,
    geometry=[Point(r.lon,r.lat) for _,r in df_p.iterrows()], crs=WGS84
).to_crs(UTM14)

resultados = []
for _, punto in gdf_p.iterrows():
    pt    = punto.geometry
    nd    = gdf_peaton[gdf_peaton.geometry.within(pt.buffer(500))].shape[0]
    nw    = gdf_wifi_s[gdf_wifi_s.geometry.within(pt.buffer(500))].shape[0]
    ns    = gdf_stops_s[gdf_stops_s.geometry.within(pt.buffer(300))].shape[0]
    col   = gdf_ri6_s[gdf_ri6_s.geometry.intersects(pt.buffer(500))]
    vl    = col['vulnerabilidad_luz'].mean() if len(col)>0 else 0.5
    inf_s = np.log(nw+ns+1)
    raw   = (nd/inf_s*vl) if inf_s>0 else nd*vl*2
    resultados.append({
        'nombre':punto['nombre'],'alcaldia':punto['alcaldia'],
        'lat':punto['lat'],'lon':punto['lon'],
        'delitos_500m':nd,'wifi_500m':nw,'stops_300m':ns,
        'vul_luz':round(vl,3),'iin_raw':round(raw,4)
    })

df_iin = pd.DataFrame(resultados)
mn, mx = df_iin.iin_raw.min(), df_iin.iin_raw.max()
df_iin['IIN']   = ((df_iin.iin_raw-mn)/(mx-mn)*100).round(1) if mx>mn else 0.0
df_iin['nivel'] = pd.cut(df_iin.IIN,[0,20,40,60,80,100],
    labels=['Muy Bajo','Bajo','Medio','Alto','Muy Alto'],include_lowest=True)

df_iin.to_parquet(GOLD / 'iin_puntos.parquet', index=False)
print('IIN calculado:')
print(df_iin[['nombre','IIN','nivel','delitos_500m','wifi_500m']].sort_values('IIN',ascending=False).to_string(index=False))


---
## 9. Sistema de peticiones y alertas automáticas <a id='s9'></a>

### Flujo de escalación

```
Ciudadano reporta queja
    │
    ▼ GeoJusticia genera folio + calcula IIN de la zona
    │
    ▼ Correo automático → Delegado de la alcaldía
    │
    ├─ Respuesta ≤ 15 días → Queja ATENDIDA → Notificación al ciudadano
    │
    └─ Sin respuesta > 15 días → Escalación automática al Director
                    │
                    └─ Sin respuesta > 15 días más → Alerta Jefatura de Gobierno
    │
    ▼ Reporte semanal (lunes 8 AM) → Todos los delegados
```


In [ ]:
# ── Base de datos SQLite ─────────────────────────────────────────────────
DB_PATH = GOLD / 'quejas_ciudadanas.db'

def get_db():
    return sqlite3.connect(DB_PATH)

def init_db():
    con = get_db()
    con.execute(
        'CREATE TABLE IF NOT EXISTS quejas ('
        'folio TEXT PRIMARY KEY,'
        'fecha_registro TEXT NOT NULL,'
        'tipo_queja TEXT NOT NULL,'
        'nombre_solicitante TEXT NOT NULL,'
        'email_solicitante TEXT NOT NULL,'
        'telefono TEXT,'
        'alcaldia TEXT NOT NULL,'
        'colonia TEXT NOT NULL,'
        'calle_referencia TEXT,'
        'latitud REAL,'
        'longitud REAL,'
        'descripcion TEXT,'
        'estatus TEXT DEFAULT "pendiente",'
        'fecha_primera_notif TEXT,'
        'fecha_escalacion TEXT,'
        'email_responsable TEXT,'
        'nivel_escalacion INTEGER DEFAULT 1,'
        'iin_zona REAL,'
        'creado_en TEXT'
        ')'
    )
    con.commit(); con.close()
    print(f'DB lista: {DB_PATH}')

init_db()


In [ ]:
# ── Motor de correo ──────────────────────────────────────────────────────
DEV_MODE = True  # True = imprimir en lugar de enviar

def enviar_correo(destinatario, asunto, cuerpo_html):
    if DEV_MODE:
        import re
        print(f'[DEV] Para: {destinatario}')
        print(f'      Asunto: {asunto}')
        texto = re.sub(r'<[^>]+>', ' ', cuerpo_html).strip()[:500]
        print(f'      Preview: {texto}...')
        return True
    try:
        msg = MIMEMultipart('alternative')
        msg['Subject'] = asunto
        msg['From']    = EMAIL_CONFIG['sender']
        msg['To']      = destinatario
        msg.attach(MIMEText(cuerpo_html, 'html', 'utf-8'))
        with smtplib.SMTP(EMAIL_CONFIG['smtp_host'], EMAIL_CONFIG['smtp_port']) as s:
            s.starttls()
            s.login(EMAIL_CONFIG['sender'], EMAIL_CONFIG['password'])
            s.sendmail(EMAIL_CONFIG['sender'], destinatario, msg.as_string())
        return True
    except Exception as e:
        print(f'Error: {e}'); return False


In [ ]:
# ── Templates de correo ──────────────────────────────────────────────────

def _header_email(titulo, subtitulo, bg='#2d6a4f'):
    return (
        '<div style="background:' + bg + ';padding:20px 28px;color:#fff">'
        '<p style="margin:0;font-size:11px;opacity:.8">GeoJusticia CDMX</p>'
        '<h2 style="margin:4px 0 0;font-size:19px">' + titulo + '</h2>'
        '<p style="margin:4px 0 0;font-size:12px;opacity:.85">' + subtitulo + '</p>'
        '</div>'
    )

def _footer_email(texto='Mensaje automatico de GeoJusticia CDMX'):
    return (
        '<div style="background:#f0efeb;padding:12px 28px;border-top:1px solid #ddd;'
        'font-size:11px;color:#888;text-align:center">' + texto + '</div>'
    )

def _tabla_kv(pares, titulo='', color='#2d6a4f'):
    filas = ''.join(
        f'<tr><td style="padding:4px 0;color:#888;width:150px">{k}</td>'
        f'<td style="padding:4px 0">{v}</td></tr>'
        for k, v in pares
    )
    encabezado = f'<h3 style="margin:0 0 10px;color:{color};font-size:14px">{titulo}</h3>' if titulo else ''
    return (
        f'<div style="background:#f8f8f6;border-left:4px solid {color};border-radius:0 8px 8px 0;'
        f'padding:16px 18px;margin:14px 0">'
        + encabezado +
        f'<table style="font-size:13px;color:#444;width:100%;border-collapse:collapse">{filas}</table>'
        '</div>'
    )

def html_queja_delegado(q):
    nivel_color = {'Muy Alto':'#c0392b','Alto':'#e07320','Medio':'#d4a017',
                   'Bajo':'#276749','Muy Bajo':'#1a56a0'}.get(q.get('nivel_iin',''),'#555')
    return (
        '<!DOCTYPE html><html lang="es"><body style="font-family:Arial;background:#f5f4f0;margin:0;padding:20px">'
        '<div style="max-width:580px;margin:0 auto;background:#fff;border-radius:10px;border:1px solid #ddd;overflow:hidden">'
        + _header_email('Nueva queja registrada', f'Alcaldia {q["alcaldia"]} · Folio {q["folio"]}')
        + '<div style="padding:22px 28px">'
        + '<p style="font-size:14px;color:#333">Se ha registrado una queja que requiere atencion. '
        + f'Tiene <strong>{DIAS_ESCALACION} dias habiles</strong> para responder; de lo contrario se escalara al Director.</p>'
        + _tabla_kv([
            ('Tipo de problema', f'<b style="color:#1a1916">{q["tipo_queja"]}</b>'),
            ('Alcaldia', q['alcaldia']),
            ('Colonia', q['colonia']),
            ('Referencia', q.get('calle_referencia','No especificada')),
            ('Descripcion', f'<i>"{q.get("descripcion","")}"</i>'),
            ('Fecha registro', q['fecha_registro']),
          ], 'Datos de la queja')
        + _tabla_kv([
            ('Nombre', q['nombre_solicitante']),
            ('Correo', q['email_solicitante']),
            ('Telefono', q.get('telefono','No proporcionado')),
          ], 'Datos del solicitante', color='#1a56a0')
        + f'<div style="background:#fff8e1;border:1px solid #ffe082;border-radius:8px;padding:12px 18px;margin:12px 0">'
        + f'<p style="margin:0;font-size:12px;color:#888">Indice de Injusticia de la zona</p>'
        + f'<p style="margin:2px 0;font-size:22px;font-weight:700;color:{nivel_color}">{q.get("iin_zona","N/D")} / 100 — {q.get("nivel_iin","Sin calcular")}</p>'
        + '</div>'
        + f'<div style="background:#fdf0ee;border:1px solid #f5a99c;border-radius:8px;padding:12px 18px;text-align:center">'
        + f'<p style="margin:0;font-size:13px;color:#c0392b;font-weight:600">Plazo maximo: {q.get("fecha_limite","15 dias habiles")}</p>'
        + '</div></div>'
        + _footer_email(f'GeoJusticia CDMX · Folio: {q["folio"]}')
        + '</div></body></html>'
    )

def html_escalacion_director(q):
    dias = q.get('dias_sin_respuesta', DIAS_ESCALACION)
    return (
        '<!DOCTYPE html><html lang="es"><body style="font-family:Arial;background:#f5f4f0;margin:0;padding:20px">'
        '<div style="max-width:580px;margin:0 auto;background:#fff;border-radius:10px;border:2px solid #c0392b;overflow:hidden">'
        + _header_email(f'ESCALACION — {dias} dias sin atencion', f'Folio {q["folio"]}', bg='#c0392b')
        + '<div style="padding:22px 28px">'
        + f'<p style="font-size:14px;color:#333;border-left:3px solid #c0392b;padding-left:12px">'
        + f'La queja folio <b>{q["folio"]}</b> lleva <b>{dias} dias</b> sin respuesta en <b>{q["alcaldia"]}</b>.</p>'
        + _tabla_kv([
            ('Tipo', q['tipo_queja']),
            ('Colonia', q['colonia']),
            ('Fecha registro', q['fecha_registro']),
            ('Solicitante', f'{q["nombre_solicitante"]} — {q["email_solicitante"]}'),
            ('Dias sin respuesta', f'<b style="color:#c0392b">{dias}</b>'),
          ], 'Queja pendiente', color='#c0392b')
        + '</div>'
        + _footer_email(f'Escalacion automatica GeoJusticia ({DIAS_ESCALACION} dias)')
        + '</div></body></html>'
    )

def html_reporte_semanal(alcaldia, stats):
    total_q = sum(s['total'] for s in stats)
    total_p = sum(s['pendientes'] for s in stats)
    filas = ''
    for s in stats:
        c = '#c0392b' if s['pendientes'] > 10 else '#276749'
        filas += (f'<tr>'
            f'<td style="padding:7px 10px;border-bottom:1px solid #eee">{s["tema"]}</td>'
            f'<td style="padding:7px 10px;border-bottom:1px solid #eee;text-align:center">{s["total"]}</td>'
            f'<td style="padding:7px 10px;border-bottom:1px solid #eee;text-align:center;color:{c};font-weight:600">{s["pendientes"]}</td>'
            f'<td style="padding:7px 10px;border-bottom:1px solid #eee;text-align:center;color:#276749">{s["resueltas"]}</td>'
            f'<td style="padding:7px 10px;border-bottom:1px solid #eee;text-align:center">{s["tasa_res"]}%</td>'
            f'</tr>')
    return (
        '<!DOCTYPE html><html lang="es"><body style="font-family:Arial;background:#f5f4f0;margin:0;padding:20px">'
        '<div style="max-width:620px;margin:0 auto;background:#fff;border-radius:10px;border:1px solid #ddd;overflow:hidden">'
        + _header_email(f'Reporte semanal — Alcaldia {alcaldia}', datetime.now().strftime('Semana del %d/%m/%Y'))
        + '<div style="padding:20px 28px">'
        + f'<div style="display:flex;gap:12px;margin-bottom:16px">'
        + f'<div style="flex:1;background:#f0fdf4;border:1px solid #a7f3d0;border-radius:8px;padding:12px;text-align:center">'
        + f'<p style="margin:0;font-size:11px;color:#888">Total quejas</p>'
        + f'<p style="margin:2px 0;font-size:26px;font-weight:700;color:#276749">{total_q}</p></div>'
        + f'<div style="flex:1;background:#fdf0ee;border:1px solid #f5a99c;border-radius:8px;padding:12px;text-align:center">'
        + f'<p style="margin:0;font-size:11px;color:#888">Pendientes</p>'
        + f'<p style="margin:2px 0;font-size:26px;font-weight:700;color:#c0392b">{total_p}</p></div></div>'
        + '<table style="width:100%;border-collapse:collapse;font-size:13px">'
        + '<thead><tr style="background:#f8f8f6">'
        + '<th style="padding:8px 10px;text-align:left;border-bottom:2px solid #ddd">Tema</th>'
        + '<th style="padding:8px 10px;text-align:center;border-bottom:2px solid #ddd">Total</th>'
        + '<th style="padding:8px 10px;text-align:center;border-bottom:2px solid #ddd">Pendientes</th>'
        + '<th style="padding:8px 10px;text-align:center;border-bottom:2px solid #ddd">Resueltas</th>'
        + '<th style="padding:8px 10px;text-align:center;border-bottom:2px solid #ddd">Tasa</th>'
        + '</tr></thead><tbody>' + filas + '</tbody></table>'
        + '</div>'
        + _footer_email('GeoJusticia CDMX · Reporte generado automaticamente cada lunes 8 AM')
        + '</div></body></html>'
    )

print('Templates de correo listos.')


In [ ]:
# ── Registrar queja y notificar delegado ─────────────────────────────────
def registrar_queja(datos):
    folio  = 'GJ-' + datetime.now().strftime('%Y%m%d') + '-' + str(uuid.uuid4())[:6].upper()
    ahora  = datetime.now().isoformat()
    fecha_limite = (datetime.now() + timedelta(days=DIAS_ESCALACION)).strftime('%d/%m/%Y')
    alc    = ALCALDIAS_MAP.get(datos.get('alcaldia',''), datos.get('alcaldia',''))

    # Calcular IIN de la zona si vienen coordenadas
    iin_zona, nivel_iin = None, 'Sin calcular'
    if datos.get('latitud') and datos.get('longitud'):
        try:
            from shapely.geometry import Point
            pt = gpd.GeoDataFrame(
                [1], geometry=[Point(datos['longitud'],datos['latitud'])], crs=WGS84
            ).to_crs(UTM14).geometry.iloc[0]
            nd  = gdf_peaton[gdf_peaton.geometry.within(pt.buffer(500))].shape[0]
            nw  = gdf_wifi_s[gdf_wifi_s.geometry.within(pt.buffer(500))].shape[0]
            ns  = gdf_stops_s[gdf_stops_s.geometry.within(pt.buffer(300))].shape[0]
            inf = np.log(nw+ns+1)
            raw = nd/inf if inf>0 else nd*2
            iin_zona  = round(min(raw*10,100),1)
            nivel_iin = ('Muy Alto' if iin_zona>=80 else 'Alto' if iin_zona>=60 else
                         'Medio'    if iin_zona>=40 else 'Bajo'  if iin_zona>=20 else 'Muy Bajo')
        except Exception as e:
            print(f'IIN no calculado: {e}')

    email_del = RESPONSABLES.get(alc,{}).get('delegado','')
    queja = {**datos,'folio':folio,'fecha_registro':ahora,'alcaldia':alc,
             'estatus':'pendiente','fecha_primera_notif':ahora,
             'email_responsable':email_del,'nivel_escalacion':1,
             'iin_zona':iin_zona,'nivel_iin':nivel_iin,'fecha_limite':fecha_limite}

    con = get_db()
    con.execute(
        'INSERT INTO quejas (folio,fecha_registro,tipo_queja,nombre_solicitante,'
        'email_solicitante,telefono,alcaldia,colonia,calle_referencia,latitud,longitud,'
        'descripcion,estatus,fecha_primera_notif,email_responsable,nivel_escalacion,iin_zona,creado_en)'
        ' VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)',
        (folio,ahora,datos.get('tipo_queja',''),datos.get('nombre_solicitante',''),
         datos.get('email_solicitante',''),datos.get('telefono',''),alc,
         datos.get('colonia',''),datos.get('calle_referencia',''),
         datos.get('latitud'),datos.get('longitud'),datos.get('descripcion',''),
         'pendiente',ahora,email_del,1,iin_zona,ahora)
    )
    con.commit(); con.close()

    if email_del:
        enviar_correo(email_del,
            f'[GeoJusticia] Nueva queja — {datos.get("tipo_queja")} en {alc} — Folio {folio}',
            html_queja_delegado(queja))

    print(f'Queja registrada: {folio}')
    print(f'  Tipo:     {datos.get("tipo_queja")}')
    print(f'  Alcaldia: {alc}')
    print(f'  IIN zona: {iin_zona} ({nivel_iin})')
    print(f'  Limite:   {fecha_limite}')
    return folio

# DEMO
folio_demo = registrar_queja({
    'tipo_queja':         'ALUMBRADO',
    'nombre_solicitante': 'Maria Gonzalez Lopez',
    'email_solicitante':  'maria.gonzalez@email.com',
    'telefono':           '55-1234-5678',
    'alcaldia':           'CUAUHTEMOC',
    'colonia':            'Tepito',
    'calle_referencia':   'Calle Tenochtitlan entre Ferrocarril de Cintura y Aztecas',
    'latitud':            19.441,
    'longitud':           -99.120,
    'descripcion':        'Luminaria sin funcionar desde hace 3 semanas. La calle queda a oscuras desde las 7pm.'
})


In [ ]:
# ── Escalacion automatica (15 dias sin respuesta) ────────────────────────
def revisar_escalaciones():
    con    = get_db()
    ahora  = datetime.now()
    limite = (ahora - timedelta(days=DIAS_ESCALACION)).isoformat()
    rows   = con.execute(
        'SELECT folio,tipo_queja,nombre_solicitante,email_solicitante,'
        'alcaldia,colonia,descripcion,fecha_registro,nivel_escalacion '
        'FROM quejas WHERE estatus="pendiente" AND nivel_escalacion=1 AND fecha_primera_notif<?',
        (limite,)
    ).fetchall()
    cols = ['folio','tipo_queja','nombre_solicitante','email_solicitante',
            'alcaldia','colonia','descripcion','fecha_registro','nivel_escalacion']
    escaladas = 0
    for row in rows:
        q    = dict(zip(cols, row))
        dias = (ahora - datetime.fromisoformat(q['fecha_registro'])).days
        q['dias_sin_respuesta'] = dias
        email_dir = RESPONSABLES.get(q['alcaldia'],{}).get('director','')
        if email_dir:
            enviar_correo(email_dir,
                f'[GeoJusticia] ESCALACION — {q["tipo_queja"]} en {q["alcaldia"]} — {dias} dias — {q["folio"]}',
                html_escalacion_director(q))
        con.execute(
            'UPDATE quejas SET nivel_escalacion=2,fecha_escalacion=?,email_responsable=? WHERE folio=?',
            (ahora.isoformat(), email_dir, q['folio'])
        )
        escaladas += 1
    con.commit(); con.close()
    print(f'Escalacion: {escaladas} quejas escaladas al director.')
    return escaladas

revisar_escalaciones()


In [ ]:
# ── Reporte semanal a delegados ──────────────────────────────────────────
def enviar_reportes_semanales():
    con = get_db()
    alcs = [r[0] for r in con.execute('SELECT DISTINCT alcaldia FROM quejas').fetchall()]
    enviados = 0
    for alc in alcs:
        rows = con.execute('SELECT tema_solicitud, estatus FROM quejas WHERE alcaldia=?',(alc,)).fetchall()
        if not rows: continue
        df_q = pd.DataFrame(rows, columns=['tema','estatus'])
        stats = []
        for tema, grp in df_q.groupby('tema'):
            tot  = len(grp)
            pend = int((grp.estatus=='pendiente').sum())
            res  = int((grp.estatus=='cerrado').sum())
            stats.append({'tema':tema,'total':tot,'pendientes':pend,
                          'resueltas':res,'tasa_res':round(res/tot*100,1)})
        stats.sort(key=lambda x: x['pendientes'], reverse=True)
        email_del = RESPONSABLES.get(alc,{}).get('delegado','')
        if email_del:
            enviar_correo(email_del,
                f'[GeoJusticia] Reporte semanal — Alcaldia {alc} — {datetime.now().strftime("%d/%m/%Y")}',
                html_reporte_semanal(alc, stats))
            enviados += 1
    con.close()
    print(f'Reportes semanales enviados: {enviados} delegados')

enviar_reportes_semanales()


In [ ]:
# ── Scheduler (produccion) ───────────────────────────────────────────────
def iniciar_scheduler():
    schedule.every().day.at('09:00').do(revisar_escalaciones)
    schedule.every().monday.at('08:00').do(enviar_reportes_semanales)
    print('Scheduler activo:')
    print('  09:00 AM diario  — Revision de escalaciones')
    print('  08:00 AM lunes   — Reporte semanal a delegados')
    while True:
        schedule.run_pending()
        time.sleep(60)

# Descomentar para activar en produccion:
# iniciar_scheduler()
print('Scheduler listo. Descomentar iniciar_scheduler() para produccion.')


In [ ]:
# ── Consultar estado de quejas ────────────────────────────────────────────
con = get_db()
df_q = pd.read_sql('SELECT alcaldia,tipo_queja,estatus,nivel_escalacion,fecha_registro,folio,iin_zona FROM quejas ORDER BY fecha_registro DESC', con)
con.close()
print(f'Total quejas en la base: {len(df_q)}')
if len(df_q) > 0:
    print()
    print('Por estatus:')
    print(df_q['estatus'].value_counts().to_string())
    print()
    print(df_q.to_string(index=False))


---
## Arquitectura del proyecto

```
FUENTES EXTERNAS (Bronze)
├── FGJ CDMX          carpetasFGJ_2024.csv          138k registros
├── C5 WiFi           wifi_gratuito_postes_c5.xlsx   13,714 postes
├── GTFS SEMOVI       stops.txt                      11,362 paradas
├── IECM 2022         colonias_iecm2022_.shp         ~2,800 poligonos
├── INEGI RI-6        ri_6.shp                       1,814 colonias
└── SUAC/Locatel      filtrado_locatel.csv           38,708 quejas
         │
         ▼  Limpieza · Normalización · Reproyección UTM 14N
         │
SILVER (GeoParquet)
├── fgj_2024.parquet           delitos clasificados (peaton/transporte/otro)
├── wifi_c5.parquet            postes con alcaldia normalizada
├── stops_gtfs.parquet         paradas con sistema decodificado
├── colonias_iecm.parquet      poligonos de colonias
├── infra_ri6.parquet          alumbrado + vulnerabilidad_luz derivada
└── locatel_2024.parquet       quejas con estatus_simple y coords
         │
         ▼  Joins espaciales · IIN · Agregaciones
         │
GOLD
├── iin_puntos.parquet         IIN normalizado [0-100] por punto de interes
├── conteos_alcaldia.parquet   metricas consolidadas por alcaldia
└── quejas_ciudadanas.db       SQLite — ciclo de vida completo de quejas
         │
         ▼  API / Portal
         │
FRONTEND
├── geojusticia_cdmx.html      Portal ciudadano con mapa + 3 capas
└── Sistema de correos         Delegado → Director → Jefatura (15 dias)
```
